In [5]:
import pandas as pd

df = pd.read_csv("../data/email_evaluation_dataset_misa.csv")
df.head()


,id,email_text,expected_action,expected_tone
0,1,Reminder: This is a gentle reminder about the ...,notify,neutral
1,2,Reminder: This is a gentle reminder about the ...,notify,neutral
2,3,Reminder: This is a gentle reminder about the ...,notify,neutral
3,4,Reminder: This is a gentle reminder about the ...,notify,neutral
4,5,Reminder: This is a gentle reminder about the ...,notify,neutral


In [8]:
source_col = 'email_text'

df['clean_text'] = (
    df[source_col]
      .fillna('')
      .astype(str)
      .str.lower()
      .str.replace(r'[^a-z\s]', '', regex=True)
      .str.strip()
)

df[[source_col, 'clean_text']].head()


,email_text,clean_text
0,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
1,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
2,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
3,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...
4,Reminder: This is a gentle reminder about the ...,reminder this is a gentle reminder about the p...


In [9]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\barat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
1,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
2,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
3,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."
4,reminder this is a gentle reminder about the p...,"[reminder, gentle, reminder, project, review, ..."


In [11]:
def triage_rule(text):
    text = text.lower()

    if "refund" in text or "urgent" in text or "password" in text:
        return "notify_human"

    if "newsletter" in text or "promotion" in text:
        return "ignore"

    return "respond_or_act"

df['triage'] = df['clean_text'].apply(triage_rule)
df[['email_text', 'triage']].head()

,email_text,triage
0,Reminder: This is a gentle reminder about the ...,respond_or_act
1,Reminder: This is a gentle reminder about the ...,respond_or_act
2,Reminder: This is a gentle reminder about the ...,respond_or_act
3,Reminder: This is a gentle reminder about the ...,respond_or_act
4,Reminder: This is a gentle reminder about the ...,respond_or_act


In [12]:
df.to_csv("../data/milestone1_output_MisaKanaujiya.csv", index=False)

In [14]:
# Check columns
print(df.columns)

# Apply triage rule
df['predicted_triage'] = df['clean_text'].apply(triage_rule)

# View result
df[['email_text', 'predicted_triage']].head()


Index(['id', 'email_text', 'expected_action', 'expected_tone', 'clean_text',
       'keywords', 'triage', 'predicted_triage'],
      dtype='object')


,email_text,predicted_triage
0,Reminder: This is a gentle reminder about the ...,respond_or_act
1,Reminder: This is a gentle reminder about the ...,respond_or_act
2,Reminder: This is a gentle reminder about the ...,respond_or_act
3,Reminder: This is a gentle reminder about the ...,respond_or_act
4,Reminder: This is a gentle reminder about the ...,respond_or_act


In [16]:
df[['email_text', 'predicted_triage']].to_csv("../data/email_evaluation_dataset_misa.csv", index=False)


In [22]:
import pandas as pd
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [17]:
#Select 100 test emails

eval_df = df.sample(n=100, random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,email_text,expected_action,expected_tone,clean_text,keywords,triage,predicted_triage
0,84,Thank you for attending the webinar. This emai...,ignore,neutral,thank you for attending the webinar this email...,"[thank, attending, webinar, email, reference, ...",respond_or_act,respond_or_act
1,54,"Dear Student, congratulations on being shortli...",respond,polite,dear student congratulations on being shortlis...,"[dear, student, congratulations, shortlisted, ...",respond_or_act,respond_or_act
2,71,"Hello, I would like to know more about the ser...",respond,polite,hello i would like to know more about the serv...,"[hello, would, like, know, services, offer, pr...",respond_or_act,respond_or_act
3,46,"Dear Student, congratulations on being shortli...",respond,polite,dear student congratulations on being shortlis...,"[dear, student, congratulations, shortlisted, ...",respond_or_act,respond_or_act
4,45,"Dear Student, congratulations on being shortli...",respond,polite,dear student congratulations on being shortlis...,"[dear, student, congratulations, shortlisted, ...",respond_or_act,respond_or_act


In [19]:
#create a column of ideal_response(if elif logic)

def ideal_response(text):
    """
    Returns the ideal response based on email content.
    """
    t = str(text).lower()  # ensure lowercase for matching

    # Security-related emails
    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return "Please escalate this security issue to the IT team immediately."

    # Marketing / promotion emails
    elif any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return "No action needed. This is a promotional email."

    # Payment / invoice / meeting emails
    elif any(k in t for k in ['invoice', 'payment', 'overdue', 'due on', 'meeting']):
        return "Please respond appropriately to the client regarding payment or schedule."

    # Default response
    else:
        return "Please read the email and respond as necessary."

In [22]:
# ensure clean_text exists before applying ideal_response
if 'clean_text' not in df.columns:
    df['clean_text'] = (
        df['email_text'].astype(str)
             .str.lower()
             .str.replace('[^a-zA-Z ]', '', regex=True)
    )

df['ideal_response'] = df['clean_text'].apply(ideal_response)
# use the existing 'triage' column (not 'triage_label')
df[['clean_text', 'triage', 'ideal_response']].head()

,clean_text,triage,ideal_response
0,reminder this is a gentle reminder about the p...,respond_or_act,Please respond appropriately to the client reg...
1,reminder this is a gentle reminder about the p...,respond_or_act,Please respond appropriately to the client reg...
2,reminder this is a gentle reminder about the p...,respond_or_act,Please respond appropriately to the client reg...
3,reminder this is a gentle reminder about the p...,respond_or_act,Please respond appropriately to the client reg...
4,reminder this is a gentle reminder about the p...,respond_or_act,Please respond appropriately to the client reg...
